# Image registration with histology images

Image registration is the process of transforming different sets of data into one coordinate system. Data may be multiple photographs, data from different sensors, times, depths, or viewpoints. It is used in computer vision, medical imaging, military automatic target recognition, and compiling and analyzing images and data from satellites. Registration is necessary in order to be able to compare or integrate the data obtained from these different measurements. [wiki](https://en.wikipedia.org/wiki/Image_registration)

![](https://www.mathworks.com/discovery/image-registration/_jcr_content/mainParsys/image_0.adapt.full.medium.jpg/1686568093060.jpg)

Some OpenCV tutortials:
- https://docs.opencv.org/4.x/dc/dc3/tutorial_py_matcher.html

In [ ]:
import os, glob
import pandas
import matplotlib.pyplot as plt

DATASET_PATH = "/kaggle/input/histology-cima-dataset"
SCALE = "scale-25pc"
# most images are in JPEG, just the 100% are PNG
IMAGE_EXTENSION = ".jpg"

## load sample image pair

In [ ]:
ls = glob.glob(os.path.join(DATASET_PATH, "*", SCALE, "*" + IMAGE_EXTENSION))
print(f"found {len(ls)} images with {SCALE}")
cases = sorted([os.path.basename(p) for p in glob.glob(os.path.join(DATASET_PATH, "*"))])
print(f"the tissue cases:\n {cases}")

In [ ]:
import random

def load_pair_images_landmarks(case: str, scale: str = SCALE):
    ls = glob.glob(os.path.join(DATASET_PATH, case, scale, "*.jpg"))
    ls += glob.glob(os.path.join(DATASET_PATH, case, scale, "*.png"))
    random.shuffle(ls)
    img1_path, img2_path = ls[:2]
    csv1_path = img1_path.replace(os.path.splitext(img1_path)[1], ".csv")
    csv2_path = img2_path.replace(os.path.splitext(img2_path)[1], ".csv")
    return (
        plt.imread(img1_path),
        plt.imread(img2_path),
        pandas.read_csv(csv1_path, index_col=0),
        pandas.read_csv(csv2_path, index_col=0)
    )

img1, img2, lnd1, lnd2 = load_pair_images_landmarks("lung-lobes_3")

## draw the perfect match with GT

In [ ]:
import cv2

kpt1 = [cv2.KeyPoint(x, y, size=1) for x, y in lnd1[['X', 'Y']].values]
kpt2 = [cv2.KeyPoint(x, y, size=1) for x, y in lnd2[['X', 'Y']].values]

matches_identity = [cv2.DMatch(i, i, 0) for i in range(len(kpt1))]
matched = cv2.drawMatches(
    img1, kpt1, img2, kpt2,
    matches1to2=matches_identity,
    outImg=None, matchesThickness=5
)

fig, ax = plt.subplots(figsize=(10, 12))
ax.imshow(matched)

# SIFT matching

In [ ]:
# Initiate SIFT detector
sift = cv2.SIFT_create()

# find the keypoints and descriptors with SIFT
kpt1, desc1 = sift.detectAndCompute(img1, None)
kpt2, desc2 = sift.detectAndCompute(img2, None)

# FLANN parameters
FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=50) # or pass empty dictionary
flann = cv2.FlannBasedMatcher(index_params, search_params)
matches = flann.knnMatch(desc1, desc2, k=2)

# Need to draw only good matches, so create a mask
matches_mask = [[0,0] for i in range(len(matches))]
# ratio test as per Lowe's paper
for i,(m,n) in enumerate(matches):
    if m.distance < 0.8 * n.distance:
        matches_mask[i] = [1,0]

In [ ]:
matched = cv2.drawMatchesKnn(
    img1, kpt1, img2, kpt2, matches,
    matchesMask=matches_mask, outImg=None
)

fig, ax = plt.subplots(figsize=(10, 12))
ax.imshow(matched)

# ORB matching

In [ ]:
# use ORB to detect keypoints and extract (binary) local invariant features
orb = cv2.ORB_create(150)

# find the keypoints and descriptors with SIFT
kpt1, desc1 = orb.detectAndCompute(img1, None)
kpt2, desc2 = orb.detectAndCompute(img2, None)

# match the features
matcher = cv2.DescriptorMatcher_create(cv2.DESCRIPTOR_MATCHER_BRUTEFORCE_HAMMING)
matches = matcher.match(desc1, desc2, None)

In [ ]:
matched = cv2.drawMatches(
    img1, kpt1, img2, kpt2,
    matches1to2=matches,
    outImg=None, matchesThickness=5
)

fig, ax = plt.subplots(figsize=(10, 12))
ax.imshow(matched)